# 01 - Benchmark TripoSR trên T4 (29 vật thể GSO)

**Mô hình**: TripoSR — checkpoint chính thức `stabilityai/TripoSR` (cùng file `model.ckpt` mà nghiên cứu C1 đã dùng — xác minh bằng SHA-256).
**Repo**: `VAST-AI-Research/TripoSR` @ commit `107cefdc244c39106fa830359024f6a2f1c78871`.

**Giao thức (giống hệt C1)**: seed 42; input = adapter `triposr` (RGB, nền xám 0.5, 512×512, đã xác minh trùng pixel với input C1 ở notebook 00);
extract: chunk 8192, resolution 256, xoay -90° trục X (z-up → y-up), Poisson depth 9, cắt density < quantile 0.001, giữ cluster lớn nhất; xuất GLB.

**Cấu trúc notebook**:
- PHẦN 1–2: kiểm tra GPU, kết nối Drive, đọc manifest
- PHẦN 3: cài đặt repo + checkpoint (xác minh hash)
- PHẦN 4: nạp mô hình + **smoke test 1 ảnh** → **DỪNG, chờ duyệt**
- PHẦN 5: batch 29 ảnh (chỉ chạy sau khi được duyệt)

**Quy tắc**: không sửa cell; cell nào lỗi → chụp màn hình gửi về; không chạy lại notebook giữa chừng nếu không được yêu cầu (resume tự động khi mở lại).

## PHẦN 1 - Kiểm tra GPU

Nếu không có CUDA → notebook **dừng ngay**. Yêu cầu GPU T4 (nếu được cấp GPU khác thì vẫn chạy nhưng phải báo lại).

In [ ]:
import subprocess, torch, platform, time

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('python:', platform.python_version())
print('torch:', torch.__version__, '| cuda:', torch.version.cuda, '| available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('KHONG CO CUDA - bam Runtime > Disconnect, chon GPU T4 roi chay lai.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
print('T4?', 'T4' in gpu_name)
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## PHẦN 2 - Mount Drive + cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, csv, json, hashlib, zipfile
import numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/nien_luan_3d_benchmark'
DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
RUNS_DIR = os.path.join(DRIVE_ROOT, 'runs')
WEIGHTS_DIR = '/content/weights_triposr'

MODEL_ID = 'triposr'
REPO_URL = 'https://github.com/VAST-AI-Research/TripoSR.git'
REPO_COMMIT = '107cefdc244c39106fa830359024f6a2f1c78871'
HF_WEIGHTS = 'stabilityai/TripoSR'
EXPECTED_CKPT_SHA256 = '429E2C6B22A0923967459DE24D67F05962B235F79CDE6B032AA7ED2FFCD970EE'
EXPECTED_CONFIG_SHA256 = 'AE7FFE1BD080B004CBA129316D3949BA12EF330C4300BC13BB484A4BF062476B'
SMOKE_ID = 'Hefty_Waste_Basket_Decorative_Bronze_85_liter'
SEED = 42

EXTRACT_RESOLUTION = 256
POISSON_DEPTH = 9
DENSITY_QUANTILE = 0.001

RUN_ID = '%s_t4_%s' % (MODEL_ID, time.strftime('%Y%m%d_%H%M%S'))
RUN_DIR = os.path.join(RUNS_DIR, RUN_ID)
MESH_DIR = os.path.join(RUN_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('RUN_ID:', RUN_ID)
print('RUN_DIR:', RUN_DIR)

In [ ]:
manifest = {}
with open(os.path.join(DATASET_DIR, 'dataset_manifest.csv'), encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        manifest[r['object_id']] = r

expected_ids = sorted(manifest.keys())
print('Manifest tren Drive:', len(expected_ids), 'objects')
assert len(expected_ids) == 29, 'Phai co 29 object trong manifest!'
assert SMOKE_ID in expected_ids, 'Thieu smoke object %s' % SMOKE_ID

adapter_dir = os.path.join(DATASET_DIR, 'adapters', 'triposr')
missing = [oid for oid in expected_ids if not os.path.exists(os.path.join(adapter_dir, oid + '.png'))]
print('Thieu adapter:', missing if missing else 'khong co')
assert not missing, 'Chay lai notebook 00 truoc khi tiep tuc!'

## PHẦN 3 - Cài đặt repo + checkpoint

Clone repo chính thức, checkout **đúng commit** ghi ở tiêu đề, cài dependency, tải checkpoint từ Hugging Face rồi **xác minh SHA-256 khớp với checkpoint C1**.

In [ ]:
if not os.path.exists('/content/TripoSR/.git'):
    !git clone --quiet {REPO_URL} /content/TripoSR
%cd /content/TripoSR
!git checkout --quiet {REPO_COMMIT}
!git log -1 --format='%H %s'

In [ ]:
!git rev-parse HEAD

In [ ]:
!pip install -q einops open3d trimesh
!pip install -q huggingface_hub
print('deps ok')

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest().upper()

ckpt_p = os.path.join(WEIGHTS_DIR, 'model.ckpt')
config_p = os.path.join(WEIGHTS_DIR, 'config.yaml')

if not (os.path.exists(ckpt_p) and sha256_file(ckpt_p) == EXPECTED_CKPT_SHA256):
    from huggingface_hub import hf_hub_download
    ckpt_p = hf_hub_download(HF_WEIGHTS, 'model.ckpt', local_dir=WEIGHTS_DIR)
    config_p = hf_hub_download(HF_WEIGHTS, 'config.yaml', local_dir=WEIGHTS_DIR)

ckpt_sha = sha256_file(ckpt_p)
config_sha = sha256_file(config_p)
print('ckpt   sha:', ckpt_sha)
print('config sha:', config_sha)
print('ckpt bytes:', os.path.getsize(ckpt_p))
assert ckpt_sha == EXPECTED_CKPT_SHA256, 'SHA-256 checkpoint KHAC voi C1 - DUNG LAI, bao ve!'
assert config_sha == EXPECTED_CONFIG_SHA256, 'SHA-256 config KHAC voi C1 - DUNG LAI, bao ve!'
print('CHECKPOINT TRUNG KHAI - ok.')

## PHẦN 4 - Nạp mô hình + SMOKE TEST

Chạy 1 ảnh đầu tiên (`Hefty_Waste_Basket_Decorative_Bronze_85_liter`) — kiểm tra mọi thứ hoạt động. Sau cell này: **DỪNG LẠI** và gửi kết quả về để duyệt.

In [ ]:
%cd /content/TripoSR
import open3d as o3d
import trimesh
from PIL import Image

import torch
torch.manual_seed(SEED)
np.random.seed(SEED)

from tsr.system import TSR

t0 = time.perf_counter()
model = TSR.from_pretrained(WEIGHTS_DIR, config_name='config.yaml', weight_name='model.ckpt')
model.to('cuda')
model.eval()
torch.cuda.empty_cache()
torch.cuda.synchronize()
model_load_time_s = round(time.perf_counter() - t0, 3)
print('model load time_s:', model_load_time_s)

In [ ]:
def extract_mesh(model, scene):
    model.renderer.set_chunk_size(8192)
    meshes = model.extract_mesh(scene, True, resolution=EXTRACT_RESOLUTION)
    raw = meshes[0]
    raw.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(np.asarray(raw.vertices, dtype=np.float64))
    o3d_mesh.triangles = o3d.utility.Vector3iVector(np.asarray(raw.faces))
    o3d_mesh.compute_vertex_normals()
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d_mesh.vertices
    pcd.normals = o3d_mesh.vertex_normals
    poisson, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=POISSON_DEPTH)
    densities = np.asarray(densities)
    poisson.remove_vertices_by_mask(densities < np.quantile(densities, DENSITY_QUANTILE))
    clusters, n_tris, _ = poisson.cluster_connected_triangles()
    n_tris = np.asarray(n_tris)
    if len(n_tris) > 0:
        largest = int(n_tris.argmax())
        poisson.remove_triangles_by_mask(np.asarray(clusters) != largest)
    return poisson

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest().upper()

def run_one(oid, mesh_dir):
    row = {'run_id': RUN_ID, 'model': MODEL_ID, 'object_id': oid,
           'commit': REPO_COMMIT, 'weights_sha256': ckpt_sha,
           'weights_source': HF_WEIGHTS, 'adapter': 'triposr', 'seed': SEED,
           'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'), 'status': 'failed',
           'success': False, 'error_message': '', 'n_vertices': '', 'n_triangles': '',
           'mesh_bytes': ''}
    try:
        t_pre = time.perf_counter()
        img = Image.open(os.path.join(adapter_dir, oid + '.png')).convert('RGB')
        row['input_sha256'] = sha256_bytes(np.asarray(img).tobytes())
        t_pre_el = round(time.perf_counter() - t_pre, 3)

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        t_inf = time.perf_counter()
        with torch.no_grad():
            scene = model(img, device='cuda')
        torch.cuda.synchronize()
        t_inf_el = round(time.perf_counter() - t_inf, 3)
        vram_peak_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1)

        t_post = time.perf_counter()
        mesh = extract_mesh(model, scene)
        glb = os.path.join(mesh_dir, oid + '.glb')
        tm = trimesh.Trimesh(vertices=np.asarray(mesh.vertices, dtype=np.float64),
                             faces=np.asarray(mesh.triangles), process=False)
        tm.export(glb)
        t_post_el = round(time.perf_counter() - t_post, 3)

        row.update({'preprocessing_time_s': t_pre_el, 'inference_time_s': t_inf_el,
                    'postprocessing_time_s': t_post_el,
                    'total_time_s': round(t_pre_el + t_inf_el + t_post_el, 3),
                    'vram_peak_mb': vram_peak_mb, 'n_vertices': int(tm.vertices.shape[0]),
                    'n_triangles': int(tm.faces.shape[0]), 'mesh_bytes': os.path.getsize(glb),
                    'status': 'success', 'success': True})
    except Exception as e:
        row['error_message'] = '%s: %s' % (type(e).__name__, e)
    return row

print('Ham da san sang.')

In [ ]:
print('SMOKE TEST object:', SMOKE_ID)
row = run_one(SMOKE_ID, MESH_DIR)
for k, v in row.items():
    print('%-24s %s' % (k, v))

exp_hash = manifest[SMOKE_ID]['adapter_triposr_sha256'].lower()
print('input hash khop manifest C1:', row.get('input_sha256', '').lower() == exp_hash)

if row['success']:
    print('SMOKE: SUCCESS')
else:
    print('SMOKE: FAIL ->', row['error_message'])
    raise SystemExit('SMOKE FAIL - DUNG LAI, gui man hinh ve.')

## ⛔ DỪNG LẠI — chờ duyệt

1. Chụp màn hình toàn bộ kết quả 2 cell trên (gồm GPU, thời gian, VRAM, SMOKE: SUCCESS).
2. Gửi về cho người hướng dẫn.
3. **KHÔNG chạy PHẦN 5 trước khi được duyệt.**

Khi được duyệt: chạy cell `PHẦN 5 - BATCH 29 ẢNH` bên dưới, để chạy cho xong, không tắt runtime.
Nếu runtime bị mất giữa chừng: mở lại notebook, chạy lại PHẦN 1→4 (cell nào có sẵn kết quả thì thôi), rồi chạy lại PHẦN 5 — resume tự động, ảnh đã xong sẽ được bỏ qua.

## PHẦN 5 - BATCH 29 ẢNH (chỉ sau khi duyệt)

- Model nạp 1 lần duy nhất (đã nạp ở PHẦN 4) — warm-up đã chạy ở smoke test, không tính vào thời gian.
- Mỗi ảnh: đo riêng `preprocessing` / `inference` / `postprocessing` / `total`, VRAM đỉnh (`max_memory_allocated`).
- Ghi CSV **ngay sau từng ảnh** → có thể resume nếu bị mất runtime.

In [ ]:
log_csv = os.path.join(RUN_DIR, 'run_log.csv')
fields = ['run_id', 'model', 'object_id', 'commit', 'weights_sha256', 'weights_source',
          'adapter', 'input_sha256', 'seed', 'timestamp', 'preprocessing_time_s',
          'inference_time_s', 'postprocessing_time_s', 'total_time_s', 'vram_peak_mb',
          'n_vertices', 'n_triangles', 'mesh_bytes', 'status', 'success', 'error_message']

done = set()
if os.path.exists(log_csv):
    with open(log_csv, encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):
            if r.get('success') == 'True':
                done.add(r['object_id'])

def append_row(row):
    new = not os.path.exists(log_csv)
    with open(log_csv, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        if new:
            w.writeheader()
        w.writerow(row)

t_batch = time.perf_counter()
for i, oid in enumerate(sorted(manifest.keys())):
    if oid in done:
        print('[%02d/%02d] SKIP (da xong): %s' % (i + 1, 29, oid))
        continue
    row = run_one(oid, MESH_DIR)
    append_row(row)
    tag = 'OK ' if row['success'] else 'ERR'
    print('[%02d/%02d] %s %s  inf=%ss  total=%ss  vram=%sMB  tris=%s' % (
        i + 1, 29, tag, oid, row.get('inference_time_s'), row.get('total_time_s'),
        row.get('vram_peak_mb'), row.get('n_triangles')))
    if not row['success']:
        print('      ->', row['error_message'])

batch_el = round(time.perf_counter() - t_batch, 3)
print('Batch time_s:', batch_el)
print('Done. Log:', log_csv)

In [ ]:
ok = 0
fail = 0
with open(log_csv, encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        if r['success'] == 'True':
            ok += 1
        else:
            fail += 1

meta = {
    'run_id': RUN_ID, 'model': MODEL_ID, 'repo': REPO_URL, 'commit': REPO_COMMIT,
    'weights': HF_WEIGHTS, 'weights_sha256': ckpt_sha, 'weights_source': 'huggingface',
    'seed': SEED, 'adapter': 'triposr_gray0.5_rgb', 'input_size': 512,
    'extract_resolution': EXTRACT_RESOLUTION, 'poisson_depth': POISSON_DEPTH,
    'density_quantile': DENSITY_QUANTILE, 'renderer_chunk_size': 8192,
    'gpu': gpu_name, 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'python': platform.python_version(),
    'time_start': t0, 'time_end': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model_load_time_s': model_load_time_s,
    'objects_total': 29, 'objects_success': ok, 'objects_failed': fail,
    'batch_time_s': batch_el,
    'note': 'same protocol as C1; single run fixed seed; no absolute comparison claims'
}
with open(os.path.join(RUN_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

zip_p = os.path.join(RUN_DIR, RUN_ID + '.zip')
with zipfile.ZipFile(zip_p, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['run_log.csv', 'metadata.json']:
        zf.write(os.path.join(RUN_DIR, fn), fn)
    for fn in sorted(os.listdir(MESH_DIR)):
        zf.write(os.path.join(MESH_DIR, fn), 'meshes/' + fn)

print('success=%d fail=%d' % (ok, fail))
print('metadata:', os.path.join(RUN_DIR, 'metadata.json'))
print('zip:', zip_p, os.path.getsize(zip_p), 'bytes')
print('CHAY XONG. Gui man hinh ket qua + ten run_id ve.')